<a href="https://colab.research.google.com/github/sakshi79/papers/blob/main/torch_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pytorch's autograd

In [1]:
import torch

In [2]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = torch.tensor([2.0, 4.0, 6.0], requires_grad=True)
z = x + y
z = torch.sum(z)
z.backward()
print(f"x: {x}, gradient: {x.grad}")
print(f"y: {y}, gradient: {y.grad}")

x: tensor([1., 2., 3.], requires_grad=True), gradient: tensor([1., 1., 1.])
y: tensor([2., 4., 6.], requires_grad=True), gradient: tensor([1., 1., 1.])


## Custom Autograd

In [3]:
import numpy as np

class Tensor:
    def __init__(self, value, parents=None, op=None, requires_grad=True):
        self.value = np.array(value, dtype=np.float32)
        self.gradient = 0.0
        self.parents = parents or []
        self.op = op

    def backward(self, grad=1.0):
        self.gradient += grad
        for parent, local_grad in self.parents:
            parent.backward(grad*local_grad(self))

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return Tensor(self.value + other.value, parents = [(self, lambda _ : np.ones_like(self.value)), (other, lambda _ : np.ones_like(other.value))], op='+')

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return Tensor(self.value - other.value, parents = [(self, lambda _: np.ones_like(self.value)), (other, lambda _ : -1*np.ones_like(other.value))], op='-')

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return Tensor(self.value*other.value, parents = [(self, lambda _: other.value), (other, lambda _: self.value)], op='*')

    def __truediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return Tensor(self.value/other.value, parents = [(self, lambda _: 1/other.value), (other, lambda _: -self.value/other.value**2)], op='/')

    def relu(self):
        return Tensor(np.maximum(self.value, 0), parents=[(self, lambda _: self.value>0)])

    def tanh(self):
        t = np.tanh(self.value)
        return Tensor(t, parents=[(self, lambda _: (1 - t**2))])

    def sigmoid(self):
        sig = 1 / (1 + np.exp(-self.value))
        return Tensor(sig, parents=[(self, lambda _: (sig*(1-sig)))])

    def sum(self, axis=None, keepdims=False):
        out = Tensor(self.value.sum(axis=axis, keepdims=keepdims), parents = [(self, lambda _: np.ones_like(self.value))])
        return out

    def __repr__(self):
        return f"Tensor (value={self.value}, grad={self.gradient})"

In [4]:
x = Tensor([1.0, 2.0, 3.0])
y = Tensor([2.0, 4.0, 6.0])
z = x + y  # Example function
z = z.sum()
z.backward()  # Backward pass to compute gradients
print(f"x: {x}")
print(f"y: {y}")

x: Tensor (value=[1. 2. 3.], grad=[1. 1. 1.])
y: Tensor (value=[2. 4. 6.], grad=[1. 1. 1.])


In [5]:
def test_autograd():
    # print("=== Forward and Backward Test ===")
    x = Tensor([1.0, -2.0, 3.0], requires_grad=True)
    y = Tensor([0.5, 4.0, -1.0], requires_grad=True)

    z = x * y + x / y - x
    a = z.relu() + x.sigmoid() + x.tanh()
    loss = a.sum()
    loss.backward()

    # print("x:", x)
    # print("y:", y)
    # print("z:", z)
    # print("a (activations):", a)
    # print("loss:", loss)

    # A pytorch program to check grads
    # Uncomment the following lines if you have PyTorch installed in your environment
    import torch
    print("=== Checking results with PyTorch ===")

    x_pt = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
    y_pt = torch.tensor([0.5, 4.0, -1.0], requires_grad=True)

    z_pt = x_pt * y_pt + x_pt / y_pt - x_pt
    a_pt = torch.relu(z_pt) + torch.sigmoid(x_pt) + torch.tanh(x_pt)

    loss_pt = a_pt.sum()

    z_pt.retain_grad()
    a_pt.retain_grad()
    loss_pt.retain_grad()

    loss_pt.backward()

    # print(x_pt.grad)
    # print(y_pt.grad)
    # print(z_pt.grad)
    # print(a_pt.grad)
    # print(loss_pt.grad)

    print("x tensor gradients match", torch.allclose(torch.tensor(x.gradient, dtype=torch.float32), x_pt.grad, atol=1e-6))
    print("y tensor gradients match", torch.allclose(torch.tensor(y.gradient, dtype=torch.float32), y_pt.grad, atol=1e-6))
    print("z tensor gradients match", torch.allclose(torch.tensor(z.gradient, dtype=torch.float32), z_pt.grad, atol=1e-6))
    print("a tensor gradients match", torch.allclose(torch.tensor(a.gradient, dtype=torch.float32), a_pt.grad, atol=1e-6))
    print("loss gradients match", torch.allclose(torch.tensor(loss.gradient, dtype=torch.float32), loss_pt.grad, atol=1e-6))

test_autograd()

=== Checking results with PyTorch ===
x tensor gradients match True
y tensor gradients match True
z tensor gradients match True
a tensor gradients match True
loss gradients match True
